In [15]:
import os
import math
import numpy as np
import chess
import chess.pgn
from collections import defaultdict, Counter

In [16]:
# Spelletjes inlezen van de lichess dataset
PGN_PATH = "Game/lichess_db_standard_rated_2017-02.pgn"     # <-- replace with your PGN file path (a subset of Lichess)
MAX_SAMPLES = 1000     # limit for prototype; reduce if memory is an issue
def parse_pgn_extract_samples(pgn_path, max_samples=MAX_SAMPLES):
    """Parse PGN file and return list of (fen_before_move, next_move_uci)."""
    samples = []
    if not os.path.exists(pgn_path):
        raise FileNotFoundError(f"PGN file not found: {pgn_path}")
    with open(pgn_path, "r", encoding="utf-8") as f:
        games = 0
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            games += 1
            board = game.board()
            for node in game.mainline():
                move = node.move
                fen_before = board.fen()
                # We store the move that was played from fen_before
                samples.append((fen_before, move.uci()))
                board.push(move)
                if len(samples) >= max_samples:
                    break
            if len(samples) >= max_samples:
                break
    print(f"Parsed {len(samples)} samples from {games} games.")
    return samples

samples = parse_pgn_extract_samples(PGN_PATH, max_samples=MAX_SAMPLES)

Parsed 1000 samples from 14 games.


In [45]:
print(samples[1])

('rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1', 'd7d5')


![image.png](Chess_naming.png)

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R
(768,)


In [44]:
# We starten opnieuw
# We gaan eerst inschatten of scoren hoe goed een zet is op basis van de database - bord positie voor de zet, de uitkomst of spel uiteindelijk gewonnen, gelijk of verloren is, en welk kleur er speelt.
# Pas als we dit hebben kunnen we kijken hoe we nu vanuit een bord spel gaan voorspellen wat de beste zet is.

In [ ]:
# pgn_stats_extract.py
import chess.pgn
from collections import defaultdict
import math

PGN_PATH = "Game/lichess_db_standard_rated_2017-02.pgn"     # <-- replace with your PGN file path (a subset of Lichess)
MAX_GAMES = 1000     # limit for prototype; reduce if memory is an issue

def iter_game_moves_with_results(pgn_path, max_games=None):
    """
    Iterate over games and yield (fen_before_move, move_uci, mover_color, result_str, white_elo, black_elo)
    result_str is one of: "1-0", "0-1", "1/2-1/2"
    """
    with open(pgn_path, "r", encoding="utf-8", errors="replace") as f:
        game_count = 0
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            game_count += 1
            headers = game.headers
            result = headers.get("Result", "")
            white_elo = headers.get("WhiteElo")
            black_elo = headers.get("BlackElo")
            # normalize elo as numbers if present, else None
            try:
                white_elo = int(white_elo) if white_elo is not None else None
            except:
                white_elo = None
            try:
                black_elo = int(black_elo) if black_elo is not None else None
            except:
                black_elo = None

            board = game.board()
            for node in game.mainline():
                move = node.move
                if move is None:
                    break
                fen_before = board.fen()
                mover_color = board.turn  # True = White, False = Black for the mover
                yield fen_before, move.uci(), mover_color, result, white_elo, black_elo
                board.push(move)

            if max_games is not None and game_count >= max_games:
                break

from collections import defaultdict

#Deze zou je wins nog kunnen versterken met de rating van de speler
def aggregate_wdl_from_pgn(pgn_path, max_games=None):
    """
    Returns a dict mapping (fen, move_uci) -> {'wins': int, 'draws': int, 'losses': int, 'count': int}
    """
    stats = defaultdict(lambda: {'wins':0, 'draws':0, 'losses':0, 'count':0})
    for fen, move_uci, mover_color, result, w_elo, b_elo in iter_game_moves_with_results(pgn_path, max_games=max_games):
        key = (fen, move_uci)
        stats[key]['count'] += 1

        # interpret result from mover's perspective
        if result == "1-0":
            winner_is_white = True
            if mover_color:  # mover is white
                stats[key]['wins'] += 1
            else:
                stats[key]['losses'] += 1
        elif result == "0-1":
            winner_is_white = False
            if not mover_color:  # mover is black
                stats[key]['wins'] += 1
            else:
                stats[key]['losses'] += 1
        elif result == "1/2-1/2":
            stats[key]['draws'] += 1
        else:
            # Unknown or abort; skip counting as valid result or handle separately
            stats[key]['count'] -= 1  # do not count ambiguous games
            # optionally continue
            continue

    return stats

stats = aggregate_wdl_from_pgn(PGN_PATH, MAX_GAMES)


In [6]:
stats

defaultdict(<function __main__.aggregate_wdl_from_pgn.<locals>.<lambda>()>,
            {('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1',
              'e2e4'): {'wins': 287, 'draws': 21, 'losses': 278, 'count': 586},
             ('rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1',
              'd7d5'): {'wins': 31, 'draws': 4, 'losses': 28, 'count': 63},
             ('rnbqkbnr/ppp1pppp/8/3p4/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2',
              'f2f3'): {'wins': 0, 'draws': 0, 'losses': 1, 'count': 1},
             ('rnbqkbnr/ppp1pppp/8/3p4/4P3/5P2/PPPP2PP/RNBQKBNR b KQkq - 0 2',
              'd5d4'): {'wins': 1, 'draws': 0, 'losses': 0, 'count': 1},
             ('rnbqkbnr/ppp1pppp/8/8/3pP3/5P2/PPPP2PP/RNBQKBNR w KQkq - 0 3',
              'd2d3'): {'wins': 0, 'draws': 0, 'losses': 1, 'count': 1},
             ('rnbqkbnr/ppp1pppp/8/8/3pP3/3P1P2/PPP3PP/RNBQKBNR b KQkq - 0 3',
              'e7e5'): {'wins': 1, 'draws': 0, 'losses': 0, 'count': 1},
            

In [8]:
import numpy as np
def compute_base_score(wins, draws, losses):
    total = wins + draws + losses
    if total == 0:
        return 0.5  # neutral when no data
    return (wins + 0.5 * draws) / total
def shrink_score(score, n, k=20, prior=0.5):
    return (n / (n + k)) * score + (k / (n + k)) * prior

# extract unique boards and moves
boards = sorted({fen for (fen, move) in stats})
moves  = sorted({move for (fen, move) in stats})

board_to_row = {fen: i for i, fen in enumerate(boards)}
move_to_col  = {move: j for j, move in enumerate(moves)}
#Matrix maken
M = np.zeros((len(boards), len(moves)), dtype=np.float32)

for (fen, move), result in stats.items():
    wins   = result["wins"]
    draws  = result["draws"]
    losses = result["losses"]
    
    base = compute_base_score(wins, draws, losses)
    n = wins + draws + losses
    
    score = shrink_score(base, n, k=20, prior=0.5)
    
    i = board_to_row[fen]
    j = move_to_col[move]
    M[i, j] = score

In [13]:
# Nu hebben we een matrix met op rijen de board posities, op de kolommen de mogelijke zetten, en in de cellen de geschatte score (0-1) voor die zet vanuit die board positie.

In [ ]:
# Nu moeten we eerst het board spel omzetten naar een vlakke tensor van 783 elementen
# 12 x 8 x8 = 768 + 15 extra features (castling rights, en passant, move count etc)

In [33]:
# piece to channel mapping: 0-5 white P,N,B,R,Q,K ; 6-11 black P,N,B,R,Q,K
PIECE_TO_IDX = {
    chess.PAWN: 0, # zegt eigenlijk 1:0
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
    chess.KING: 5
}
def fen_to_tensor_flat(fen,enforce_color=200):
    """
    Convert FEN -> 8x8x12 binary tensor flattened to 768 vector.
    Order: rows 8->1, files a->h, channels 12.
    """
    board = chess.Board(fen)
    #print(board)
    tensor = np.zeros((12, 8, 8), dtype=np.float32)
    for sq in chess.SQUARES: # loopt door het spelbord van a1 - b1, c1,... 8=a2; ..63=h8
        piece = board.piece_at(sq)
        if piece:
            row = 7 - chess.square_rank(sq)  # rank 8 at row 0
            col = chess.square_file(sq) # dit geeft voor a =0, b=1, c=2, ..., h=7
            base = PIECE_TO_IDX[piece.piece_type] #geef voor een stuk een nummer maar mapt ze van 0-5 ipv 1-6
            if piece.color == chess.WHITE:
                ch = base
            else:
                ch = base + 6
            tensor[ch, row, col] = 1.0
    flat_pieces= tensor.ravel()  # shape (768,) -flattened 8x8x12 matrix
        # 2. Side to move (N value - based on enforce color)
    # --------------------------
    if board.turn == chess.WHITE:
        stm=np.ones(enforce_color, dtype=np.float32)
    else:
        stm = np.zeros(enforce_color, dtype=np.float32)
    # 3. Castling rights (4 binary values)
    # -------------------------------------
    castling = np.array([
        1.0 if board.has_kingside_castling_rights(chess.WHITE) else 0.0,
        1.0 if board.has_queenside_castling_rights(chess.WHITE) else 0.0,
        1.0 if board.has_kingside_castling_rights(chess.BLACK) else 0.0,
        1.0 if board.has_queenside_castling_rights(chess.BLACK) else 0.0,
    ], dtype=np.float32)
        # ------------------------------------------
    # 4. En passant file (8 one-hot features)
    # ------------------------------------------
    ep_vec = np.zeros(8, dtype=np.float32)
    if board.ep_square is not None:
        file_idx = chess.square_file(board.ep_square)  # 0–7
        ep_vec[file_idx] = 1.0
    return np.concatenate([flat_pieces, stm, castling, ep_vec])
# An example of a 8x8x12 tensor flattened to 768 +enforce_color+4+8 vector
a=fen_to_tensor_flat(samples[1][0])
print(a)
print(a.shape)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 1. 1. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [43]:
#function to quickly pick a board of the the pgn file or database
def random_position_fast(path):
    """
    Fast method: randomly seek inside PGN file to find a random game,
    then pick a random move inside it.
    """

    import os

    filesize = os.path.getsize(path)

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        # --- 1. Random seek into file ---
        f.seek(random.randint(0, filesize - 1))

        # --- 2. Read until start of next game ---
        line = f.readline()
        while not line.startswith("[Event "):
            line = f.readline()
            if line == "":
                f.seek(0)
                line = f.readline()
        game = chess.pgn.read_game(f)

    # --- 3. Extract random position from game ---
    node = game
    positions = [node.board()]

    while node.variations:
        node = node.variations[0]
        positions.append(node.board())

    return random.choice(positions)

board = random_position_fast(PGN_PATH)
print(board.fen())
query_fen = board.fen()
query_next_move=board.peek().uci()
print(query_next_move)


1rbq1rk1/4bpp1/pp1p3p/2pPp3/2P1B3/1P2PN1P/P1Q2PP1/R4RK1 b - - 0 17
h2h3


In [60]:
#Nemen we gewoon een zet uit de "trainingmatrix"
import random
query=random.choice(samples)
query_fen=query[0]
query_next_move=query[1]
print(query_fen)
print(query_next_move)

rn5r/pbb1kpp1/7p/3PP3/1PN5/P2P4/4KPPP/RN3B1R b - - 0 19
b7d5


In [61]:
X = np.array([fen_to_tensor_flat(fen) for fen in boards])  # shape = (num_rows, 782+enforce_color+8+4)
from sklearn.preprocessing import normalize
X_norm = normalize(X, axis=1)

q = fen_to_tensor_flat(query_fen)
q_norm = q / np.linalg.norm(q)

from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(q_norm.reshape(1, -1), X_norm)[0]
k = 30
idx = np.argsort(-similarities)[:k]
nearest_boards = [boards[i] for i in idx]

neighbor_scores = M[idx].mean(axis=0)
top_moves_idx = np.argsort(-neighbor_scores)[:3]
best_moves = [moves[i] for i in top_moves_idx]

In [62]:
print(best_moves)

['e8e7', 'd6c7', 'b7d5']


In [ ]:
# te bekijken hoe we dit met train en test data zouden kunnen beoordelen: bv in TOP1 of TOP5 voorstellen -> zit de werkelijke zet erin => we geven een % terug van de testset.
# of omgekeerd we geven een score 1/(positie aan de voorspelling versus werkelijke zet) => dus 1/1 als voorspelde zet ook de werkelijke was, 1/5 als voorspelde zet op 5 stond etc. We kunnen dan het gemiddelde nemen over de testset.
#  